In [ ]:
!pip install --upgrade huggingface_hub langchain-core

In [ ]:
# Install dependencies
!pip install langgraph langchain langchain-community langchain_text_splitters
!pip install chromadb sentence-transformers
!pip install langchain-huggingface
!pip install huggingface_hub

In [ ]:
"""
=============================================================
AGENTIC AI: Sistem RAG & Pembuatan Konten Belajar Siswa
=============================================================
VERSI FINAL - Sudah ditest dan diperbaiki semua error
=============================================================

Cara pakai:
1. Dapatkan token gratis di https://huggingface.co/settings/tokens
2. Ganti HF_TOKEN di bawah
3. pip install langgraph langchain langchain-core langchain-community \
       langchain-huggingface chromadb sentence-transformers huggingface_hub
4. python agentic_learning.py
=============================================================
"""

import os
from typing import TypedDict, Annotated, List, Optional, Any

from huggingface_hub import InferenceClient

# LangGraph
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages

# LangChain Core
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import (
    BaseMessage, AIMessage, HumanMessage, SystemMessage
)
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.documents import Document

# Embeddings & Vector Store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ================================================================
# ⚙️ KONFIGURASI - GANTI TOKEN DI SINI
# ================================================================

HF_TOKEN = os.getenv("HF_TOKEN", "")  # <-- GANTI!

# ✅ Model yang SUDAH DITEST bisa jalan gratis:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# Alternatif jika Qwen tidak bisa:
# MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
# MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"  # v0.2 bukan v0.3!


# ================================================================
# 🤖 CUSTOM LLM - Wrapper untuk HuggingFace InferenceClient
# ================================================================

class HFChatModel(BaseChatModel):
    """
    Custom Chat LLM yang menggunakan HuggingFace InferenceClient.

    Mengapa custom?
    - langchain-huggingface sering error karena perubahan API HuggingFace
    - Dengan custom wrapper, kita kontrol penuh cara memanggil API
    - Lebih stabil dan mudah di-debug
    """

    client: Any = None
    model_id: str = MODEL_ID
    temperature: float = 0.3
    max_tokens: int = 2048

    class Config:
        arbitrary_types_allowed = True

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.client = InferenceClient(
            model=self.model_id,
            token=HF_TOKEN,
        )
        # Test koneksi
        try:
            test = self.client.chat_completion(
                messages=[{"role": "user", "content": "test"}],
                max_tokens=5,
            )
            print(f"✅ LLM terhubung: {self.model_id}")
        except Exception as e:
            print(f"❌ Gagal konek ke {self.model_id}: {e}")
            raise

    @property
    def _llm_type(self) -> str:
        return "hf-chat"

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        **kwargs
    ) -> ChatResult:
        """Kirim messages ke HuggingFace dan terima response"""

        # Konversi LangChain messages → HuggingFace format
        hf_messages = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                hf_messages.append({"role": "system", "content": msg.content})
            elif isinstance(msg, HumanMessage):
                hf_messages.append({"role": "user", "content": msg.content})
            elif isinstance(msg, AIMessage):
                hf_messages.append({"role": "assistant", "content": msg.content})
            else:
                hf_messages.append({"role": "user", "content": str(msg.content)})

        # Panggil API
        response = self.client.chat_completion(
            messages=hf_messages,
            max_tokens=self.max_tokens,
            temperature=self.temperature,
            stop=stop or [],
        )

        content = response.choices[0].message.content
        return ChatResult(
            generations=[ChatGeneration(message=AIMessage(content=content))]
        )


def get_llm(temperature=0.3, max_tokens=2048):
    """Factory function untuk LLM"""
    return HFChatModel(
        model_id=MODEL_ID,
        temperature=temperature,
        max_tokens=max_tokens,
    )


def get_embeddings():
    """Embedding model lokal (100% gratis, tanpa API)"""
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
    )


# ================================================================
# 📊 STATE - Data yang mengalir di seluruh workflow
# ================================================================

class AgentState(TypedDict):
    """
    State = 'memori bersama' yang bisa dibaca/tulis oleh semua agent.

    Bayangkan seperti papan tulis besar di ruang rapat:
    - Router menulis "tipe request"
    - Retriever menulis "dokumen yang ditemukan"
    - Quality Checker menulis "skor kualitas"
    - Semua agent bisa melihat apa yang ditulis agent lain
    """
    user_query: str
    student_level: str
    subject: str
    request_type: str
    retrieved_documents: List[str]
    documents_relevant: bool
    rag_answer: str
    curriculum_plan: str
    learning_content: str
    quiz_content: str
    quality_score: int
    quality_feedback: str
    revision_count: int
    max_revisions: int
    final_output: str
    messages: Annotated[list, add_messages]


# ================================================================
# 📚 KNOWLEDGE BASE - Database Pengetahuan Sekolah
# ================================================================

class KnowledgeBase:
    """
    Berisi materi pelajaran yang digunakan RAG untuk menjawab.

    Dalam produksi nyata, ini bisa diganti dengan:
    - PDF buku pelajaran yang di-embed
    - Database soal ujian
    - Materi dari Kemendikbud
    """

    DOCUMENTS = [
        Document(
            page_content="""
            Teorema Pythagoras menyatakan bahwa dalam segitiga siku-siku,
            kuadrat sisi miring (hipotenusa) sama dengan jumlah kuadrat
            kedua sisi lainnya. Rumusnya: c² = a² + b²

            Dimana:
            - c = hipotenusa (sisi terpanjang)
            - a dan b = kedua sisi siku-siku

            Contoh: Jika a = 3 cm dan b = 4 cm, maka:
            c² = 3² + 4² = 9 + 16 = 25, jadi c = √25 = 5 cm

            Triple Pythagoras yang umum: (3,4,5), (5,12,13), (8,15,17)

            Penerapan: mengukur jarak diagonal, arsitektur, navigasi.
            """,
            metadata={"subject": "matematika", "topic": "pythagoras"}
        ),
        Document(
            page_content="""
            Persamaan Kuadrat: ax² + bx + c = 0, dimana a ≠ 0

            Cara menyelesaikan:
            1. Faktorisasi: cari (x - p)(x - q) = 0
            2. Melengkapkan kuadrat sempurna
            3. Rumus ABC: x = (-b ± √(b² - 4ac)) / 2a

            Diskriminan (D) = b² - 4ac:
            - D > 0 → dua akar real berbeda
            - D = 0 → dua akar real sama
            - D < 0 → tidak ada akar real

            Contoh: x² - 5x + 6 = 0
            Faktorisasi: (x-2)(x-3) = 0 → x = 2 atau x = 3
            """,
            metadata={"subject": "matematika", "topic": "persamaan_kuadrat"}
        ),
        Document(
            page_content="""
            Hukum Newton tentang Gerak:

            Hukum I (Inersia): Benda tetap diam atau bergerak lurus
            beraturan jika tidak ada gaya luar.
            Contoh: penumpang terdorong ke depan saat mobil rem mendadak.

            Hukum II (F = ma): Percepatan berbanding lurus dengan gaya
            dan berbanding terbalik dengan massa.
            F = gaya (Newton), m = massa (kg), a = percepatan (m/s²)

            Hukum III (Aksi-Reaksi): Setiap aksi menimbulkan reaksi
            yang sama besar tapi berlawanan arah.
            Contoh: roket meluncur karena gas terdorong ke bawah.
            """,
            metadata={"subject": "fisika", "topic": "hukum_newton"}
        ),
        Document(
            page_content="""
            Fotosintesis: proses tumbuhan membuat makanan menggunakan
            cahaya matahari.

            Rumus: 6CO₂ + 6H₂O + Cahaya → C₆H₁₂O₆ + 6O₂
            (Karbondioksida + Air + Cahaya → Glukosa + Oksigen)

            Terjadi di KLOROPLAS yang mengandung KLOROFIL (pigmen hijau).

            Tahapan:
            1. Reaksi Terang (membran tilakoid): menyerap cahaya,
               memecah air, menghasilkan ATP & NADPH
            2. Reaksi Gelap/Siklus Calvin (stroma): mengikat CO₂,
               menghasilkan glukosa

            Faktor: intensitas cahaya, CO₂, suhu, ketersediaan air.
            """,
            metadata={"subject": "biologi", "topic": "fotosintesis"}
        ),
        Document(
            page_content="""
            Proklamasi Kemerdekaan Indonesia: 17 Agustus 1945

            Dibacakan oleh Ir. Soekarno didampingi Drs. Mohammad Hatta
            di Jalan Pegangsaan Timur No. 56, Jakarta.

            Kronologi:
            - 6 Agustus: Bom atom Hiroshima
            - 9 Agustus: Bom atom Nagasaki
            - 14 Agustus: Jepang menyerah
            - 16 Agustus: Peristiwa Rengasdengklok
            - 17 Agustus: Proklamasi dibacakan

            Teks proklamasi diketik Sayuti Melik.
            Bendera Merah Putih dijahit Ibu Fatmawati.
            """,
            metadata={"subject": "sejarah", "topic": "proklamasi"}
        ),
        Document(
            page_content="""
            Sistem Tata Surya: Matahari sebagai pusat, 8 planet:
            1. Merkurius - terkecil, terdekat Matahari
            2. Venus - terpanas, disebut Bintang Kejora
            3. Bumi - satu-satunya yang ada kehidupan
            4. Mars - planet merah, gunung Olympus Mons
            5. Jupiter - terbesar, Bintik Merah Besar
            6. Saturnus - terkenal dengan cincinnya
            7. Uranus - rotasi miring 90 derajat
            8. Neptunus - terjauh, berwarna biru

            Juga ada: asteroid, komet, planet kerdil (Pluto).
            Bulan = satelit alami Bumi.
            """,
            metadata={"subject": "ipa", "topic": "tata_surya"}
        ),
    ]

    def __init__(self):
        print("📚 Membangun Knowledge Base...")
        self.embeddings = get_embeddings()
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500, chunk_overlap=100
        )
        docs = splitter.split_documents(self.DOCUMENTS)
        self.vectorstore = Chroma.from_documents(
            documents=docs,
            embedding=self.embeddings,
            collection_name="school_kb",
        )
        print(f"   ✅ {len(docs)} chunks tersimpan di vector store")

    def search(self, query: str, k: int = 3) -> List[Document]:
        return self.vectorstore.similarity_search(query, k=k)


# ================================================================
# 🤖 AGENT NODES - Setiap node = satu agent spesialis
# ================================================================

class AgentNodes:
    """
    10 Agent yang bekerja sama dalam workflow:

    1. Router         → Klasifikasi permintaan
    2. Retriever      → Cari dokumen relevan (RAG)
    3. Grader         → Nilai relevansi dokumen
    4. Answer Gen     → Buat jawaban (RAG)
    5. Curriculum     → Rancang outline materi
    6. Content Gen    → Tulis materi belajar
    7. Quiz Gen       → Buat soal latihan
    8. Combiner       → Gabungkan semua konten
    9. Quality Check  → Evaluasi kualitas
    10. Revision      → Perbaiki berdasarkan feedback
    """

    def __init__(self):
        print("\n🤖 Menginisialisasi Agent...")
        self.llm = get_llm()
        self.kb = KnowledgeBase()
        print("   ✅ Semua agent siap\n")

    def _chat(self, system: str, user: str) -> str:
        """Helper: kirim system + user message ke LLM"""
        response = self.llm.invoke([
            SystemMessage(content=system),
            HumanMessage(content=user),
        ])
        return response.content.strip()

    # ─────────────────────────────────────
    # 1️⃣  ROUTER AGENT
    # ─────────────────────────────────────
    def router_agent(self, state: AgentState) -> dict:
        """
        Mengklasifikasikan: apakah siswa BERTANYA atau minta DIBUATKAN materi?

        Ini adalah "decision maker" pertama dalam workflow.
        Agentic AI bisa MEMILIH jalur berbeda berdasarkan input.
        """
        print("\n" + "─"*50)
        print("🔀 [1. ROUTER AGENT]")

        query = state["user_query"]

        result = self._chat(
            system="Klasifikasikan permintaan ke: question (bertanya/minta penjelasan) atau create_content (minta dibuatkan materi/modul/ringkasan). Jawab SATU kata saja.",
            user=f"Permintaan: {query}"
        )

        # Fallback: keyword detection
        create_kw = ["buatkan", "buat", "susun", "rancang", "siapkan",
                     "materi", "modul", "rangkuman", "ringkasan"]
        is_create = any(k in query.lower() for k in create_kw)

        if "create" in result.lower() or "content" in result.lower() or is_create:
            req_type = "create_content"
        else:
            req_type = "question"

        print(f"   Permintaan : {query[:60]}...")
        print(f"   Keputusan  : {req_type}")

        return {
            "request_type": req_type,
            "messages": [AIMessage(content=f"Router → {req_type}")]
        }

    # ─────────────────────────────────────
    # 2️⃣  RETRIEVER AGENT
    # ─────────────────────────────────────
    def retriever_agent(self, state: AgentState) -> dict:
        """
        Mencari dokumen relevan dari Knowledge Base (vector store).

        Ini adalah komponen "R" dari RAG (Retrieval-Augmented Generation).
        Agent ini MENGGUNAKAN TOOL (vector search) - ciri khas Agentic AI.
        """
        print("\n" + "─"*50)
        print("🔍 [2. RETRIEVER AGENT]")

        docs = self.kb.search(state["user_query"], k=3)
        texts = [d.page_content.strip() for d in docs]

        for i, t in enumerate(texts):
            preview = t.replace('\n', ' ')[:70]
            print(f"   📄 Dok {i+1}: {preview}...")

        return {
            "retrieved_documents": texts,
            "messages": [AIMessage(content=f"Ditemukan {len(texts)} dokumen")]
        }

    # ─────────────────────────────────────
    # 3️⃣  GRADER AGENT
    # ─────────────────────────────────────
    def grader_agent(self, state: AgentState) -> dict:
        """
        Menilai: apakah dokumen yang ditemukan RELEVAN?

        Ini adalah SELF-REFLECTION - sistem mengevaluasi
        kualitas informasi sebelum menggunakannya.
        """
        print("\n" + "─"*50)
        print("⚖️  [3. GRADER AGENT]")

        docs = state.get("retrieved_documents", [])
        if not docs:
            print("   ❌ Tidak ada dokumen")
            return {"documents_relevant": False}

        combined = "\n---\n".join(docs[:3])
        result = self._chat(
            system="Tentukan apakah dokumen RELEVAN untuk menjawab pertanyaan. Jawab HANYA: ya atau tidak.",
            user=f"Pertanyaan: {state['user_query']}\n\nDokumen:\n{combined}"
        )

        relevant = any(w in result.lower() for w in ["ya", "yes", "relevan"])
        print(f"   Relevan: {'✅ Ya' if relevant else '❌ Tidak'}")

        return {
            "documents_relevant": relevant,
            "messages": [AIMessage(content=f"Relevansi: {relevant}")]
        }

    # ─────────────────────────────────────
    # 4️⃣  ANSWER GENERATOR
    # ─────────────────────────────────────
    def answer_generator(self, state: AgentState) -> dict:
        """
        Membuat jawaban yang jelas sesuai level siswa.

        Menggunakan konteks dari Retriever (jika relevan)
        atau pengetahuan umum LLM (jika tidak relevan).
        """
        print("\n" + "─"*50)
        print("💡 [4. ANSWER GENERATOR]")

        level = state.get("student_level", "SMP")
        docs = state.get("retrieved_documents", [])
        relevant = state.get("documents_relevant", False)

        context = "\n".join(docs) if relevant and docs else "Tidak ada konteks khusus."

        level_guide = {
            "SD": "Gunakan bahasa sangat sederhana, analogi dari kehidupan anak-anak, banyak contoh visual.",
            "SMP": "Gunakan bahasa jelas & terstruktur, contoh step-by-step, penjelasan konsep.",
            "SMA": "Gunakan bahasa akademis, penjelasan mendalam, contoh aplikasi nyata.",
        }

        answer = self._chat(
            system=f"""Kamu adalah guru {level} yang sabar dan kreatif.
{level_guide.get(level, level_guide['SMP'])}
Gunakan emoji untuk membuat jawaban lebih menarik.
Jika ada konteks referensi, gunakan sebagai dasar jawaban.""",
            user=f"""Konteks referensi:
{context}

Pertanyaan siswa {level}: {state['user_query']}

Berikan jawaban yang lengkap, terstruktur, dan mudah dipahami siswa {level}:"""
        )

        print(f"   ✅ Jawaban: {len(answer)} karakter")

        return {
            "rag_answer": answer,
            "final_output": answer,
            "messages": [AIMessage(content="Jawaban dibuat")]
        }

    # ─────────────────────────────────────
    # 5️⃣  CURRICULUM PLANNER
    # ─────────────────────────────────────
    def curriculum_planner(self, state: AgentState) -> dict:
        """
        Merencanakan outline materi SEBELUM konten ditulis.

        Ini menunjukkan PLANNING - Agentic AI merencanakan
        dulu sebelum bertindak, bukan langsung menulis.
        """
        print("\n" + "─"*50)
        print("📋 [5. CURRICULUM PLANNER]")

        level = state.get("student_level", "SMP")
        subject = state.get("subject", "umum")

        # Cari referensi
        docs = self.kb.search(state["user_query"], k=2)
        ref = "\n".join([d.page_content for d in docs]) if docs else "Tidak ada referensi khusus."

        plan = self._chat(
            system=f"Kamu perancang kurikulum ahli untuk siswa {level}.",
            user=f"""Rancang outline materi belajar:

Topik: {state['user_query']}
Tingkat: {level}
Mata Pelajaran: {subject}

Referensi: {ref}

Buat outline yang mencakup:
1. 🎯 Tujuan Pembelajaran (3-5 poin spesifik)
2. 📋 Prasyarat (apa yang harus sudah dipahami)
3. 📖 Struktur Materi (sub-topik yang akan dibahas, urutan logis)
4. ⏰ Estimasi Waktu Belajar
5. 📝 Metode Evaluasi yang cocok"""
        )

        print(f"   ✅ Kurikulum dirancang")

        return {
            "curriculum_plan": plan,
            "messages": [AIMessage(content="Kurikulum dirancang")]
        }

    # ─────────────────────────────────────
    # 6️⃣  CONTENT GENERATOR
    # ─────────────────────────────────────
    def content_generator(self, state: AgentState) -> dict:
        """
        Menulis materi belajar berdasarkan rencana kurikulum.

        Ini menunjukkan KOLABORASI ANTAR AGENT -
        Content Generator menggunakan output dari Curriculum Planner.
        """
        print("\n" + "─"*50)
        print("📝 [6. CONTENT GENERATOR]")

        level = state.get("student_level", "SMP")
        plan = state.get("curriculum_plan", "")

        docs = self.kb.search(state["user_query"], k=3)
        ref = "\n".join([d.page_content for d in docs]) if docs else ""

        content = self._chat(
            system=f"Kamu guru kreatif yang menulis materi belajar menarik untuk siswa {level}.",
            user=f"""Berdasarkan rencana kurikulum:
{plan}

Dan referensi:
{ref}

Tulis materi belajar lengkap tentang: {state['user_query']}

Format yang HARUS diikuti:
📚 **JUDUL MATERI**

🎯 **Tujuan Pembelajaran**
(apa yang akan dikuasai setelah belajar ini)

📖 **Penjelasan Materi**
(jelaskan konsep dengan bahasa {level}, bagi menjadi sub-bagian)
(setiap sub-bagian berikan contoh konkret)

💡 **Poin Penting yang Harus Diingat**
(rangkum poin-poin kunci)

🌍 **Penerapan dalam Kehidupan Sehari-hari**
(contoh nyata penggunaan konsep ini)

📌 **Ringkasan**
(rangkuman singkat seluruh materi)"""
        )

        print(f"   ✅ Materi: {len(content)} karakter")

        return {
            "learning_content": content,
            "messages": [AIMessage(content="Materi ditulis")]
        }

    # ─────────────────────────────────────
    # 7️⃣  QUIZ GENERATOR
    # ─────────────────────────────────────
    def quiz_generator(self, state: AgentState) -> dict:
        """
        Membuat soal latihan berdasarkan materi yang sudah ditulis.

        Agent ini menggunakan OUTPUT dari agent sebelumnya -
        menunjukkan data flow antar agent melalui shared state.
        """
        print("\n" + "─"*50)
        print("📝 [7. QUIZ GENERATOR]")

        level = state.get("student_level", "SMP")
        content = state.get("learning_content", "")[:2000]

        quiz = self._chat(
            system=f"Kamu pembuat soal ujian untuk siswa {level}. Buat soal yang menguji pemahaman, bukan hafalan.",
            user=f"""Berdasarkan materi ini:
{content}

Buatkan soal latihan:

📝 **SOAL LATIHAN**

**A. Pilihan Ganda (5 soal)**
(4 opsi: a, b, c, d - hanya 1 yang benar)

**B. Isian Singkat (3 soal)**
(jawaban 1-3 kata)

**C. Essay (2 soal)**
(soal yang mendorong berpikir kritis)

🔑 **KUNCI JAWABAN & PEMBAHASAN**
(jawaban + penjelasan singkat mengapa jawaban tersebut benar)"""
        )

        print(f"   ✅ Soal latihan dibuat")

        return {
            "quiz_content": quiz,
            "messages": [AIMessage(content="Soal dibuat")]
        }

    # ─────────────────────────────────────
    # 8️⃣  CONTENT COMBINER
    # ─────────────────────────────────────
    def content_combiner(self, state: AgentState) -> dict:
        """Menggabungkan semua bagian menjadi satu paket lengkap"""
        print("\n" + "─"*50)
        print("📦 [8. CONTENT COMBINER]")

        level = state.get("student_level", "SMP")

        combined = f"""
{'═'*60}
📚 PAKET MATERI BELAJAR — Tingkat {level}
{'═'*60}

{'─'*60}
📋 RENCANA PEMBELAJARAN
{'─'*60}
{state.get('curriculum_plan', '-')}

{'─'*60}
📖 MATERI BELAJAR
{'─'*60}
{state.get('learning_content', '-')}

{'─'*60}
📝 LATIHAN & EVALUASI
{'─'*60}
{state.get('quiz_content', '-')}

{'═'*60}
✨ Selamat belajar! Semangat! 💪
{'═'*60}
"""
        print(f"   ✅ Digabungkan: {len(combined)} karakter")

        return {
            "final_output": combined,
            "revision_count": state.get("revision_count", 0),
            "messages": [AIMessage(content="Konten digabungkan")]
        }

    # ─────────────────────────────────────
    # 9️⃣  QUALITY CHECKER
    # ─────────────────────────────────────
    def quality_checker(self, state: AgentState) -> dict:
        """
        Mengevaluasi kualitas output dan memberi skor.

        Ini adalah SELF-EVALUATION / REFLECTION:
        - Sistem menilai outputnya SENDIRI
        - Memutuskan apakah perlu perbaikan
        - Ini yang membedakan Agentic AI dari AI biasa!
        """
        print("\n" + "─"*50)
        print("🔍 [9. QUALITY CHECKER]")

        level = state.get("student_level", "SMP")
        output = state.get("final_output", "")[:3000]

        result = self._chat(
            system="""Kamu quality assurance untuk materi pendidikan.
Evaluasi dan berikan skor 1-10.

WAJIB gunakan format ini:
SKOR: [angka]
FEEDBACK: [penjelasan]""",
            user=f"Evaluasi materi untuk siswa {level}:\n\n{output}"
        )

        # Parse skor
        score = 7  # default
        try:
            for line in result.split('\n'):
                if 'SKOR' in line.upper():
                    digits = ''.join(c for c in line if c.isdigit())
                    if digits:
                        score = min(max(int(digits[:2]), 1), 10)
                    break
        except:
            pass

        rev = state.get("revision_count", 0) + 1
        max_rev = state.get("max_revisions", 2)

        print(f"   📊 Skor     : {score}/10")
        print(f"   🔄 Revisi   : {rev}/{max_rev}")

        return {
            "quality_score": score,
            "quality_feedback": result,
            "revision_count": rev,
            "messages": [AIMessage(content=f"Skor: {score}/10")]
        }

    # ─────────────────────────────────────
    # 🔟 REVISION AGENT
    # ─────────────────────────────────────
    def revision_agent(self, state: AgentState) -> dict:
        """
        Memperbaiki konten berdasarkan feedback Quality Checker.

        ITERATIVE IMPROVEMENT: Agent ini bisa dipanggil berkali-kali
        sampai kualitas output memenuhi standar.
        """
        print("\n" + "─"*50)
        print("✏️  [10. REVISION AGENT]")

        level = state.get("student_level", "SMP")
        output = state.get("final_output", "")[:3000]
        feedback = state.get("quality_feedback", "")

        revised = self._chat(
            system=f"Kamu editor materi pendidikan untuk siswa {level}. Perbaiki berdasarkan feedback.",
            user=f"""MATERI SAAT INI:
{output}

FEEDBACK PERBAIKAN:
{feedback}

Perbaiki materi agar lebih berkualitas.
Pertahankan yang sudah baik, perbaiki yang kurang:"""
        )

        print(f"   ✅ Direvisi: {len(revised)} karakter")

        return {
            "final_output": revised,
            "messages": [AIMessage(content="Konten direvisi")]
        }


# ================================================================
# 🔀 ROUTING FUNCTIONS - Keputusan conditional
# ================================================================

def route_request(state: AgentState) -> str:
    """Router: pertanyaan → RAG, buat konten → Content Pipeline"""
    return "create_content" if state.get("request_type") == "create_content" else "question"


def should_revise(state: AgentState) -> str:
    """
    Keputusan: apakah output sudah cukup baik?

    Ini adalah INTI dari Agentic Loop:
    - Skor ≥ 7 ATAU sudah revisi maks → terima (selesai)
    - Skor < 7 DAN belum revisi maks → revisi (loop kembali)
    """
    score = state.get("quality_score", 7)
    revisions = state.get("revision_count", 0)
    max_rev = state.get("max_revisions", 2)

    if score >= 7 or revisions >= max_rev:
        print(f"   ✅ DITERIMA (skor={score}, revisi={revisions})")
        return "accept"
    else:
        print(f"   🔄 REVISI LAGI (skor={score}, revisi={revisions})")
        return "revise"


# ================================================================
# 🏗️ BUILD GRAPH - Merakit semua agent menjadi workflow
# ================================================================

def build_graph():
    """
    Membangun LangGraph workflow.

    Visualisasi:

    START → ROUTER ─┬─ question ──→ RETRIEVER → GRADER → ANSWER_GEN ─┐
                     │                                                 │
                     └─ create ───→ CURRICULUM → CONTENT → QUIZ ──→   │
                                    PLANNER      GEN       GEN        │
                                                            │         │
                                                        COMBINER     │
                                                            │         │
                                                            └────┬────┘
                                                                 │
                                                          QUALITY_CHECKER
                                                           │          │
                                                        accept     revise
                                                           │          │
                                                          END    REVISION ─┐
                                                                    │      │
                                                                    └──────┘
                                                                  (loop back
                                                                   to QC)
    """
    print("🏗️  Membangun Workflow Graph...")

    agents = AgentNodes()
    wf = StateGraph(AgentState)

    # Tambah semua node
    wf.add_node("router", agents.router_agent)
    wf.add_node("retriever", agents.retriever_agent)
    wf.add_node("grader", agents.grader_agent)
    wf.add_node("answer_generator", agents.answer_generator)
    wf.add_node("curriculum_planner", agents.curriculum_planner)
    wf.add_node("content_generator", agents.content_generator)
    wf.add_node("quiz_generator", agents.quiz_generator)
    wf.add_node("content_combiner", agents.content_combiner)
    wf.add_node("quality_checker", agents.quality_checker)
    wf.add_node("revision", agents.revision_agent)

    # === EDGES ===

    # Start → Router
    wf.add_edge(START, "router")

    # Router → (Question path ATAU Content path)
    wf.add_conditional_edges("router", route_request, {
        "question": "retriever",
        "create_content": "curriculum_planner",
    })

    # --- Question/RAG path ---
    wf.add_edge("retriever", "grader")
    wf.add_edge("grader", "answer_generator")
    wf.add_edge("answer_generator", "quality_checker")

    # --- Content creation path ---
    wf.add_edge("curriculum_planner", "content_generator")
    wf.add_edge("content_generator", "quiz_generator")
    wf.add_edge("quiz_generator", "content_combiner")
    wf.add_edge("content_combiner", "quality_checker")

    # --- Quality loop (shared) ---
    wf.add_conditional_edges("quality_checker", should_revise, {
        "accept": END,
        "revise": "revision",
    })
    wf.add_edge("revision", "quality_checker")

    graph = wf.compile()
    print("✅ Workflow Graph siap!\n")
    return graph


# ================================================================
# 🎓 MAIN APPLICATION
# ================================================================

class LearningAssistant:
    """Interface utama untuk berinteraksi dengan sistem"""

    def __init__(self):
        print("🚀 Memulai Agentic AI Learning Assistant...\n")
        self.graph = build_graph()

    def ask(self, query: str, level: str = "SMP", subject: str = "umum") -> str:
        """Jalankan query melalui agent workflow"""

        print(f"\n{'═'*60}")
        print(f"🎓 PERMINTAAN SISWA")
        print(f"   Pertanyaan : {query}")
        print(f"   Tingkat    : {level}")
        print(f"   Mapel      : {subject}")
        print(f"{'═'*60}")

        initial_state: AgentState = {
            "user_query": query,
            "student_level": level,
            "subject": subject,
            "request_type": "",
            "retrieved_documents": [],
            "documents_relevant": False,
            "rag_answer": "",
            "curriculum_plan": "",
            "learning_content": "",
            "quiz_content": "",
            "quality_score": 0,
            "quality_feedback": "",
            "revision_count": 0,
            "max_revisions": 2,
            "final_output": "",
            "messages": [],
        }

        print("\n🏃 Menjalankan Workflow...\n")
        result = self.graph.invoke(initial_state)

        print(f"\n{'═'*60}")
        print(f"🏁 WORKFLOW SELESAI")
        print(f"   Tipe Request : {result.get('request_type')}")
        print(f"   Skor Kualitas: {result.get('quality_score')}/10")
        print(f"   Total Revisi : {result.get('revision_count')}")
        print(f"{'═'*60}")

        return result.get("final_output", "Terjadi kesalahan.")

    def interactive(self):
        """Mode interaktif"""
        print("""
╔═══════════════════════════════════════════════════╗
║  🎓 ASISTEN BELAJAR INTERAKTIF                   ║
║                                                   ║
║  Contoh pertanyaan:                               ║
║  • "Jelaskan teorema Pythagoras"                 ║
║  • "Apa itu Hukum Newton?"                       ║
║  • "Buatkan materi tentang fotosintesis"         ║
║  • "Buatkan modul belajar tentang tata surya"    ║
║                                                   ║
║  Ketik 'quit' untuk keluar                       ║
╚═══════════════════════════════════════════════════╝
        """)

        while True:
            query = input("\n🎓 Pertanyaan: ").strip()

            if query.lower() in ['quit', 'exit', 'q', 'keluar']:
                print("\n👋 Sampai jumpa! Selamat belajar! 💪\n")
                break
            if not query:
                print("   ❓ Ketik pertanyaan kamu...")
                continue

            level = input("📚 Tingkat (SD/SMP/SMA) [SMP]: ").strip().upper()
            if level not in ["SD", "SMP", "SMA"]:
                level = "SMP"

            subject = input("📖 Mata pelajaran [umum]: ").strip() or "umum"

            result = self.ask(query, level, subject)

            print(f"\n{'═'*60}")
            print("📚 HASIL:")
            print(f"{'═'*60}")
            print(result)
            print(f"{'═'*60}\n")


# ================================================================
# 🚀 ENTRY POINT
# ================================================================

if __name__ == "__main__":
    print("""
╔══════════════════════════════════════════╗
║  🤖 AGENTIC AI LEARNING SYSTEM         ║
║     Powered by LangGraph + HuggingFace  ║
║                                          ║
║  Pilih mode:                             ║
║  1. Demo Pertanyaan (RAG Pipeline)      ║
║  2. Demo Pembuatan Konten               ║
║  3. Mode Interaktif                      ║
╚══════════════════════════════════════════╝
    """)

    choice = input("Pilihan (1/2/3): ").strip()
    assistant = LearningAssistant()

    if choice == "1":
        print("\n📌 Demo: Siswa bertanya tentang Pythagoras (RAG)\n")
        result = assistant.ask(
            query="Jelaskan tentang teorema Pythagoras dan berikan contoh soal",
            level="SMP",
            subject="matematika"
        )
        print(f"\n{'═'*60}\n📚 HASIL AKHIR:\n{'═'*60}\n{result}")

    elif choice == "2":
        print("\n📌 Demo: Siswa minta dibuatkan materi fotosintesis\n")
        result = assistant.ask(
            query="Buatkan materi belajar lengkap tentang fotosintesis",
            level="SMP",
            subject="biologi"
        )
        print(f"\n{'═'*60}\n📚 HASIL AKHIR:\n{'═'*60}\n{result}")

    elif choice == "3":
        assistant.interactive()

    else:
        print("Pilihan tidak valid, masuk mode interaktif...")
        assistant.interactive()

/tmp/ipykernel_3729/4267740889.py:59: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class HFChatModel(BaseChatModel):



╔══════════════════════════════════════════╗
║  🤖 AGENTIC AI LEARNING SYSTEM         ║
║     Powered by LangGraph + HuggingFace  ║
║                                          ║
║  Pilih mode:                             ║
║  1. Demo Pertanyaan (RAG Pipeline)      ║
║  2. Demo Pembuatan Konten               ║
║  3. Mode Interaktif                      ║
╚══════════════════════════════════════════╝
    
Pilihan (1/2/3): 1
🚀 Memulai Agentic AI Learning Assistant...

🏗️  Membangun Workflow Graph...

🤖 Menginisialisasi Agent...
✅ LLM terhubung: Qwen/Qwen2.5-7B-Instruct
📚 Membangun Knowledge Base...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   ✅ 12 chunks tersimpan di vector store
   ✅ Semua agent siap

✅ Workflow Graph siap!


📌 Demo: Siswa bertanya tentang Pythagoras (RAG)


════════════════════════════════════════════════════════════
🎓 PERMINTAAN SISWA
   Pertanyaan : Jelaskan tentang teorema Pythagoras dan berikan contoh soal
   Tingkat    : SMP
   Mapel      : matematika
════════════════════════════════════════════════════════════

🏃 Menjalankan Workflow...


──────────────────────────────────────────────────
🔀 [1. ROUTER AGENT]
   Permintaan : Jelaskan tentang teorema Pythagoras dan berikan contoh soal...
   Keputusan  : question

──────────────────────────────────────────────────
🔍 [2. RETRIEVER AGENT]
   📄 Dok 1: Teorema Pythagoras menyatakan bahwa dalam segitiga siku-siku,         ...
   📄 Dok 2: Teorema Pythagoras menyatakan bahwa dalam segitiga siku-siku,         ...
   📄 Dok 3: Teorema Pythagoras menyatakan bahwa dalam segitiga siku-siku,         ...

──────────────────────────────────────────────────
⚖️  [3. 